In [1]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [2]:
input_data = "input_files/mathematical_reasoning_0126.csv"
output_data = "output_files/mathematical_reasoning_0126.xlsx"

In [3]:
# ## Open AI API
# os.environ["OPENAI_API_KEY"] = "your open ai api"
from dotenv import load_dotenv

load_dotenv()

True

## common functions

In [4]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [5]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [6]:
"""
    This function is different a bit among different tasks, here is for the mathematical reasoning task
"""
def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []

    
    for res in results:
        segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        potential_seg_names = []

        ## 
        for seg_name in real_segmentation_names:
            normalized_segmentation_name = segmentation_name.lower()
            normalized_seg_name = seg_name.lower()
            if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
                potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        
        ##special for mathematical reasoning task
        answers = {"extracted_value": res["error_tags"].get("reported_value"), "calculated_value": res["error_tags"].get("correct_value"), "is_correct": "Yes"}
        special_inputs = {"tag": res["error_tags"].get("tag"), "period": res["error_tags"].get("period")}

        for segmentation_name_ in potential_seg_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True
            ## check the instance document
            check_tag = special_inputs.get("tag")
            
            try:
                if check_tag in temp_ins:
                    pass
                elif check_tag.replace(":","") in temp_ins:
                    pass
                elif check_tag.replace(":","_") in temp_ins:
                    pass
                else:
                    is_appear = False
               
                if is_appear:
                    parse = True
                    cal = temp_cal
                    pre = temp_pre
                    defn = temp_defn
                    lab = temp_lab
                    ins = temp_ins
                    sch = temp_sch
                    seg_path = os.path.join(folder_name, segmentation_name_)
                    break
            except Exception as e:
                print(f"Errors {e}")
            
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(special_inputs)
        outputByLLM.append(res)
        
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for mathematical reasoning task

In [7]:
mr_126_df = pd.read_csv(input_data)

In [8]:
mr_126_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0126.9590,"2022-03-09 20:00:03,338 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
1,DQC.US.0126.9590,"2022-03-09 20:00:03,339 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
2,DQC.US.0126.9590,"2022-03-04 20:32:57,578 [DQC.US.0126.9590] The...",2022-03-05,ERROR,DQC,126,1766352,"Carrier EQ, LLC",2022-01-21 19:05:00,0001553350-22-000052,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_06B_USD_1_iso42...,10-K,http://www.sec.gov/Archives/edgar/data/1766352...,2022-01-24,2022,413266,https://www.sec.gov/Archives/edgar/data/176635...,7372,10k-airfox-20210930
3,DQC.US.0126.9590,"2022-03-04 20:32:57,579 [DQC.US.0126.9590] The...",2022-03-05,ERROR,DQC,126,1766352,"Carrier EQ, LLC",2022-01-21 19:05:00,0001553350-22-000052,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_06B_USD_1_iso42...,10-K,http://www.sec.gov/Archives/edgar/data/1766352...,2022-01-24,2022,413266,https://www.sec.gov/Archives/edgar/data/176635...,7372,10k-airfox-20210930
4,DQC.US.0126.9590,"2022-03-09 20:10:36,329 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,712515,ELECTRONIC ARTS INC.,2022-02-08 16:06:00,0000712515-22-000003,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/712515/...,2022-02-08,2022,416925,https://www.sec.gov/Archives/edgar/data/712515...,7372,10q-ea-20211231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,DQC.US.0126.9590,"2024-11-18 01:36:13,415 [DQC.US.0126.9590] The...",2024-11-18,ERROR,DQC,126,1936756,"NEXT BRIDGE HYDROCARBONS, INC.",2024-11-13 17:06:00,0001199835-24-000499,US GAAP 2024,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/1936756...,2024-11-13,2024,780614,https://www.sec.gov/Archives/edgar/data/193675...,1311,10q-nbh-20240930
105,DQC.US.0126.9590,"2024-11-18 01:32:15,561 [DQC.US.0126.9590] The...",2024-11-18,ERROR,DQC,126,1309082,"Camber Energy, Inc.",2024-11-13 17:00:00,0001477932-24-007116,US GAAP 2024,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1309082...,2024-11-13,2024,780573,https://www.sec.gov/Archives/edgar/data/130908...,1311,10q-cei-20240930
106,DQC.US.0126.9590,"2024-10-28 16:04:04,616 [DQC.US.0126.9590] The...",2024-10-28,ERROR,DQC,126,205402,"GRAYBAR ELECTRIC COMPANY, INC.",2024-10-28 15:33:00,0000205402-24-000039,US GAAP 2024,XBRL document created by Certent Disclosure Ma...,10-Q,http://www.sec.gov/Archives/edgar/data/205402/...,2024-10-28,2024,771631,https://www.sec.gov/Archives/edgar/data/205402...,5063,10q-c402-20240930
107,DQC.US.0126.9590,"2024-10-11 14:46:53,604 [DQC.US.0126.9590] The...",2024-10-11,ERROR,DQC,126,1407704,"LINGERIE FIGHTING CHAMPIONSHIPS, INC.",2024-10-11 13:54:00,0001640334-24-001502,US GAAP 2024,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1407704...,2024-10-11,2024,767858,https://www.sec.gov/Archives/edgar/data/140770...,7900,10q-boty-20240630


## LLM judge assisstant

In [9]:
Client = OpenAI()

In [15]:
mr_126_system_prompt = """You are given a DQC validation message. Your task is to extract the following key information:

1. **Statement Name**: Extract the name of the financial statement mentioned in the message. If it appears in a phrase like "000004 - Statement - CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)", ignore any numeric prefixes and the phrase "- Statement -". Only return the clean name of the statement, such as "CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)".

2. **Erroneous Total Element Tag Information**:
   Extract detailed information about the total element (`us-gaap:` tag) that is reported incorrectly. Specifically, you should extract:

   - `tag`: The total element tag, as indicated by the **“Total Element”** field in the message.
   - `period`: The reporting period associated with this tag, as indicated by the **“Total period”** field.
   - `reported_value`: The value that was reported in the filing, taken from the **“Total Value”** field.
   - `correct_value`: The expected value, i.e., the **sum of the child components** as defined in the calculation linkbase.
     - This is usually explicitly stated in a sentence like: *“The sum of these child components is...”*
     - You may cross-validate this by summing the listed child component values in the message if provided.
   - `reason`: Provide a concise explanation of the error — why the reported value is considered incorrect — based directly on the message content.

Only include this tag if all of the following are true:
- The message identifies a total element with incorrect calculation
- Both reported and correct values are stated or inferable
- The error occurs in a specified reporting period

Return your result as structured JSON in the following format:
```json
{
  "statement_name": "...",
  "error_tags": 
    {
      "tag": "...",
      "period": "...",
      "reported_value": "...",
      "correct_value": "...",
      "reason": "..."
    }
}
"""

In [16]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": mr_126_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [17]:
res = []
for _, row in tqdm(mr_126_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name"))

    ## record the folder name
    response["folder_name"] = folder_name
    ## record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

109it [03:09,  1.74s/it]


In [24]:
res[4]

{'statement_name': 'Condensed Consolidated Statements Of Cash Flows',
 'error_tags': {'tag': 'us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease',
  'period': '2021-04-01 to 2021-12-31',
  'reported_value': '-2,590,000,000',
  'correct_value': '5,490,000,000',
  'reason': 'The reported value of the total element does not equal the sum of its child components as defined in the calculation linkbase. The sum of the child components is 5,490,000,000, while the reported value is -2,590,000,000, indicating an inconsistency that renders the financial statement mathematically inaccurate.'},
 'folder_name': '10q-ea-20211231',
 'segmentation_name': 'CondensedConsolidatedStatementsOfCashFlows'}

In [23]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_mr_126_df = construct_annotation(mr_126_df, res, task_name="mathematical_reasoning")

Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable


In [25]:
new_mr_126_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0126.9590,"2022-03-09 20:00:03,338 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,None,None,None,None,None,{'tag': 'us-gaap:NetCashProvidedByUsedInOperat...,"{'extracted_value': '-1,313,536', 'calculated_...",None,False,"{'statement_name': 'Statements of Cash Flows',..."
1,DQC.US.0126.9590,"2022-03-09 20:00:03,339 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,None,None,None,None,None,{'tag': 'us-gaap:NetCashProvidedByUsedInFinanc...,"{'extracted_value': '707,500', 'calculated_val...",None,False,"{'statement_name': 'Statements of Cash Flows',..."
2,DQC.US.0126.9590,"2022-03-04 20:32:57,578 [DQC.US.0126.9590] The...",2022-03-05,ERROR,DQC,126,1766352,"Carrier EQ, LLC",2022-01-21 19:05:00,0001553350-22-000052,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:NetCashProvidedByUsedInOperat...,"{'extracted_value': '-4,771,555', 'calculated_...",10k-airfox-20210930/StatementsOfCashFlows,True,"{'statement_name': 'STATEMENTS OF CASH FLOWS',..."
3,DQC.US.0126.9590,"2022-03-04 20:32:57,579 [DQC.US.0126.9590] The...",2022-03-05,ERROR,DQC,126,1766352,"Carrier EQ, LLC",2022-01-21 19:05:00,0001553350-22-000052,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"{'tag': 'us-gaap:OperatingExpenses', 'period':...","{'extracted_value': '11,635,141', 'calculated_...",10k-airfox-20210930/StatementsOfOperationsAndC...,True,{'statement_name': 'STATEMENTS OF OPERATIONS A...
4,DQC.US.0126.9590,"2022-03-09 20:10:36,329 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,712515,ELECTRONIC ARTS INC.,2022-02-08 16:06:00,0000712515-22-000003,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:CashAndCashEquivalentsPeriodI...,"{'extracted_value': '-2,590,000,000', 'calcula...",10q-ea-20211231/CondensedConsolidatedStatement...,True,{'statement_name': 'Condensed Consolidated Sta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,DQC.US.0126.9590,"2024-11-18 01:36:13,415 [DQC.US.0126.9590] The...",2024-11-18,ERROR,DQC,126,1936756,"NEXT BRIDGE HYDROCARBONS, INC.",2024-11-13 17:06:00,0001199835-24-000499,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"{'tag': 'us-gaap:NonoperatingIncomeExpense', '...","{'extracted_value': '998,557', 'calculated_val...",10q-nbh-20240930/CondensedConsolidatedStatemen...,True,{'statement_name': 'CONDENSED CONSOLIDATED STA...
105,DQC.US.0126.9590,"2024-11-18 01:32:15,561 [DQC.US.0126.9590] The...",2024-11-18,ERROR,DQC,126,1309082,"Camber Energy, Inc.",2024-11-13 17:00:00,0001477932-24-007116,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:NetCashProvidedByUsedInOperat...,"{'extracted_value': '-1,454,022', 'calculated_...",10q-cei-20240930/CondensedConsolidatedStatemen...,True,{'statement_name': 'Condensed Consolidated Sta...
106,DQC.U

In [26]:
new_mr_126_df.to_excel(output_data,index=False)